# CoalBlend AI — Demo End-to-End (tanpa Claude Desktop)

Notebook ini memanggil langsung fungsi Python di balik 3 MCP server
(`mcp_servers/spreadsheet_server.py`, `blend_server.py`, `dashboard_server.py`) —
tool `@mcp.tool()` tetap bisa dipanggil sebagai fungsi Python biasa, jadi cocok untuk
dites/didebug di Jupyter sebelum dipakai lewat Claude Desktop.

Alur: **spreadsheet template → ingest → blend (3 skenario) → render dashboard HTML**

**Sebelum run:** install dependency dulu (sekali saja):
```bash
pip install -r ../mcp_servers/requirements.txt
pip install -r ../requirements.txt
```

In [9]:
!pip install -r ../mcp_servers/requirements.txt
!pip install -r ../requirements.txt

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 24.2 -> 26.2.1
[notice] To update, run: C:\Program Files\Python312\python.exe -m pip install --upgrade pip


Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 24.2 -> 26.2.1
[notice] To update, run: C:\Program Files\Python312\python.exe -m pip install --upgrade pip


In [10]:
import sys
from pathlib import Path

# Sesuaikan kalau folder project dipindah dari lokasi ini.
PROJECT_ROOT = Path(r"C:\Users\Riza\Downloads\coalblend_mvp_v3\coalblend_mvp1")
sys.path.insert(0, str(PROJECT_ROOT / "mcp_servers"))

import spreadsheet_server as sp
import blend_server as bl
import dashboard_server as db

print("Modules loaded OK")

Modules loaded OK


## 1. Ingest — baca spreadsheet template (coalblend-data)

Bisa ganti `workbook_path` ke file real Anda sendiri (asal nama sheet & kolom sama
persis dengan template: `opening_stock`, `coal_getting`, `demand_deal`, `plan_kapasitas`).

In [11]:
ingested = sp.ingest_coal_data_from_workbook("fixtures/coalblend_input_template.xlsx")

print("status:", ingested["status"])
print("alerts:", ingested["alerts"])
ingested["data"]

status: VALID
alerts: []


{'status': 'VALID',
 'opening_stock': {'Lati': {'volume_mt': 200000.0, 'cv_gar': 5000.0},
  'Binsua': {'volume_mt': 190000.0, 'cv_gar': 4950.0},
  'Sambarata': {'volume_mt': 250000.0, 'cv_gar': 5900.0},
  'Gurimbang': {'volume_mt': 50000.0, 'cv_gar': 3880.0},
  'BMO3': {'volume_mt': 10000.0, 'cv_gar': 3350.0}},
 'coal_getting': {'Lati': {'volume_mt': 600000.0, 'cv_gar': 5000.0},
  'Binsua': {'volume_mt': 1000000.0, 'cv_gar': 4950.0},
  'Sambarata': {'volume_mt': 350000.0, 'cv_gar': 5900.0},
  'Gurimbang': {'volume_mt': 650000.0, 'cv_gar': 3880.0},
  'BMO3': {'volume_mt': 200000.0, 'cv_gar': 3350.0}},
 'demand_deal': {'Eboni': {'volume_deal_mt': 70000.0},
  'Mahoni B': {'volume_deal_mt': 195000.0},
  'Agathis': {'volume_deal_mt': 65000.0},
  'Sungkai': {'volume_deal_mt': 230000.0},
  'Sungkai B': {'volume_deal_mt': 290000.0},
  'CV 4600': {'volume_deal_mt': 260000.0},
  'CV 4200': {'volume_deal_mt': 325000.0},
  'CV 3800': {'volume_deal_mt': 65000.0}},
 'plan_kapasitas': {'Lati': {'kapa

## 2. Blend — solve 3 skenario (coalblend-blend)

In [12]:
result = bl.run_blending(ingested["data"])

print("status:", result["status"])
for sc in result["scenarios"]:
    print(f"  {sc['id']:>2} {sc['nama']:<32} revenue={sc['revenue']:>14,.2f}  delta={sc['delta_vs_baseline']:>+12,.2f}")

status: ok
   A Max Revenue (Baseline)           revenue= 24,225,858.52  delta=       +0.00
   B Quality-Safe (buffer +25 GAR)    revenue= 24,108,010.17  delta= -117,848.35
   C Evakuasi Stok Site Kritis        revenue= 23,947,502.67  delta= -278,355.85


In [13]:
baseline = bl.get_baseline_info()
print("expected baseline revenue:", baseline["baseline_validasi_revenue"])
assert abs(result["scenarios"][0]["revenue"] - baseline["baseline_validasi_revenue"]) < 1, "revenue tidak cocok baseline tervalidasi!"
print("OK — Opsi A cocok dengan baseline tervalidasi tests/test_pipeline.py")

expected baseline revenue: 24225858.52
OK — Opsi A cocok dengan baseline tervalidasi tests/test_pipeline.py


## 3. Dashboard — render HTML statis (coalblend-dashboard)

In [14]:
rendered = db.render_review_html(ingested["data"], out_path="output/dashboard_notebook_demo.html")
print("status:", rendered["status"])
print("path:", rendered["path"])
rendered["scenarios_summary"]

status: ok
path: C:\Users\Riza\Downloads\coalblend_mvp_v3\coalblend_mvp1\output\dashboard_notebook_demo.html


[{'id': 'A',
  'nama': 'Max Revenue (Baseline)',
  'revenue': 24225858.521157492,
  'delta': 0.0},
 {'id': 'B',
  'nama': 'Quality-Safe (buffer +25 GAR)',
  'revenue': 24108010.174976993,
  'delta': -117848.3461804986},
 {'id': 'C',
  'nama': 'Evakuasi Stok Site Kritis',
  'revenue': 23947502.67058308,
  'delta': -278355.85057441145}]

In [15]:
# Tampilkan langsung di notebook
from IPython.display import IFrame
IFrame(src=rendered["path"], width="100%", height=600)

## 4. (Opsional) Terbitkan Report Final untuk 1 opsi yang dipilih

In [16]:
report = db.render_report_html(
    ingested["data"],
    scenario_id="A",
    validator="partogi.luhut",
    catatan="demo notebook end-to-end",
    out_path="output/report_notebook_demo.html",
)
print(report)

{'status': 'ok', 'path': 'C:\\Users\\Riza\\Downloads\\coalblend_mvp_v3\\coalblend_mvp1\\output\\report_notebook_demo.html', 'scenario_id': 'A', 'revenue': 24225858.521157492}
